### SIMD Matrix Multiplication – Optimized Kernel

We implement an optimized matrix multiplication of the form:

$$
C = A \times B
$$

using:

- SIMD instructions (SSE, AVX2, AVX512)
- Fused Multiply-Add (FMA)
- Blocking to optimize cache usage
- Unrolled micro-kernels
- Specialized fast-paths for small fixed-size matrices

---

#### Blocking Strategy

To reduce memory bandwidth pressure and improve temporal locality, we divide the computation into sub-blocks of size `block_size × block_size`.

Let $A \in \mathbb{R}^{m \times k}$ and $B \in \mathbb{R}^{k \times n}$.

We compute $C = AB$ block by block:
- Iterate over blocks `ii`, `jj` of $C$,
- Transpose $B$ once beforehand to access its columns as rows (improves memory access),
- Compute inner products using unrolled SIMD kernels.

---

#### SIMD Microkernel

We define a `microkernel4<UN>()` that unrolls the computation of 4 columns of $C$ in parallel:

```cpp
template<int UN>
void microkernel4(size_t nCols, const K* a, const K* const* b, reg* sum)
```
For each SIMD-aligned block of k, we do:

-Load a SIMD register av from row a
-Broadcast scalar values from 4 rows of b
-Accumulate with fmadd into 4 accumulators sum[0..3]

The main inner loop looks like:

```cpp
for (; k + W <= nCols; k += W) {
    reg av = Simd::load(a + k);
    #pragma unroll(UN)
    for (int x = 0; x < UN; ++x)
        sum[x] = Simd::fmadd(av, Simd::set1(b[x][k]), sum[x]);
}
```

Tail Loop for Scalar Remainder
After the SIMD-aligned part, we complete the remaining elements ```(if cols % SIMD_WIDTH != 0):```

Horizontal Reduction and Storing
After computing the 4 partial sums, we reduce and store them:

```cpp
K total0 = Simd::horizontal_add(sum[0]);
// ...
Simd::store(&C[i * cols + j], Simd::set(total3, total2, total1, total0));
```

If the memory is aligned (using pointer mask check), we stream the result directly to memory to avoid polluting caches:

```cpp
if (((uintptr_t)&result.data[i * cols + j] & 31) == 0)
    Simd::stream(...);
else
    Simd::store(...);
```

## Fast Paths for Small Fixed Matrices
For fixed sizes like $N = 4, 8, 16$, we define specialized SIMD routines:
```cpp
template<size_t N>
Matrix mul_mat_NxN(const Matrix& mat);
```

This version:

Loads each row of $A$ and each column of $B$ into SIMD registers

Computes: $C_{ij} := \sum_{k=1}^{p} A_{ik} B_{kj}, \quad \text{for all } 1 \leq i \leq m,\, 1 \leq j \leq n.$

fully in registers using ```Simd::mul + Simd::horizontal_add.```



## Performances 

On an Intel Core i7-10700 (AVX2, 8c/16t) in fp32:
```
Matrix size : 8192 x 8192
Time        : 1.014 s
GFLOP/s     : 1078.7
Sample C(0,0): 2025.988
```

in fp64:

```
Matrix size : 8192 x 8192
Time        : 1.677  s
GFLOP/s     : 655.518
Sample C(0,0): 2037.029

```

This is close to peak CPU performance for AVX2 with well-optimized memory access.

| Technique       | Description                                    |
| --------------- | ---------------------------------------------- |
| Blocking        | Keeps working sets in L1/L2 cache              |
| SIMD            | Parallel processing (4, 8, 16 floats at once)  |
| FMA             | Reduces latency and instruction count          |
| Unrolled Kernel | Processes 4 outputs of C in parallel           |
| Tail Handling   | Scalar fallback for non-aligned parts          |
| Fast Path (NxN) | Specialized code for small square matrices     |
| Streaming Store | Avoids polluting caches when alignment permits |






### Low-Level Architecture Mapping of SIMD Matrix Multiplication

To achieve near-peak performance on modern CPUs, we must map the algorithm onto the architecture’s hardware in a cache- and pipeline-friendly way.

---

#### 1. Overview of the CPU Execution Pipeline (Simplified)

```
┌──────────────┐
│   L1 Cache   │ ← stores matrix blocks (per core)
└─────┬────────┘
      │ loads/stores
┌─────▼────────┐
│ Vector Units │ ← AVX2/AVX512 registers (YMM/ZMM)
│   Registers  │ ← SIMD microkernel uses these
└─────┬────────┘
      │ fused multiply-add
┌─────▼────────┐
│  FMA Units   │ ← 2 to 4 per core max (limited resource)
└──────────────┘
```

---

#### 2. Unrolling Strategy and Resource Usage

We distinguish two levels of **loop unrolling**:

**Outer Unroll (AVX)** → vector width

* Operates on `Simd::width` elements at once
* AVX2: 8 floats per register (`__m256`), AVX512: 16 (`__m512`)
* Example loop:

  ```cpp
  for (k += Simd::width) { ... Simd::load(a + k); }
  ```
* Goal: saturate **vector registers** (YMM/ZMM)

**Inner Unroll (FMA)** → number of dot products in parallel (e.g., `UN = 4`)

* Microkernel computes 4 columns of \$C\$ at once:

  ```cpp
  reg sum[4] = { ... };
  for (x = 0; x < 4; ++x)
      sum[x] = fmadd(...);
  ```
* Goal: saturate the **FMA units** per core
* Limitation: typical Intel core has only **2 or 3 FMA pipelines**
* Over-unrolling → **pipeline congestion** and register spilling (pollutes cache)

---

#### 3. What Happens If You Mismanage Unrolling?

| Problem                 | Cause                                          | Effect                               |
| ----------------------- | ---------------------------------------------- | ------------------------------------ |
| Cache pollution         | Unrolled loops reuse too many different blocks | Evicts useful tiles from L1          |
| FMA unit saturation     | Too many dot products in parallel              | Reduces IPC (instructions per cycle) |
| Register spilling       | `sum[UN]` too large for physical registers     | Forces RAM traffic (slowdown)        |
| Execution unit conflict | Overlapping FMA and integer instructions       | Pipeline stalls and resource wait    |

---

#### 4. Optimal Balance Strategy

Let:

* \$W\$ = SIMD width (e.g., 8 for AVX2, 16 for AVX512)
* \$U\$ = number of FMA accumulators (e.g., 4)
* \$B\$ = block size for cache tiling

Then the balance is:

* Choose \$U \approx\$ number of FMA units (typically 2–4)
* Choose \$W\$ = native SIMD width (auto-detected)
* Choose \$B\$ such that working set fits in L1 (e.g., \$B = 64\$ or \$128\$)

---

#### 5. Example on AVX2 CPU (Intel Core i7-10700)

* 2 FMA units / core
* 8-wide SIMD (AVX2)
* Microkernel unroll \$U = 4\$
* Fast path for 4 outputs of \$C\_{i,j..j+3}\$ in parallel
* Each inner loop feeds `Simd::fmadd()` with a row broadcast × vector

```
┌──────────── A[i][k..] row ─────────────┐
│  [ a0 a1 a2 a3 a4 a5 a6 a7 ]           │  ← loaded as __m256 (AVX2)
└────────────────────────────────────────┘
                 │
                 ▼
       ┌──────────────┐     ┌──────────────┐     ┌──────────────┐     ┌──────────────┐
       │  FMA Unit 0  │     │  FMA Unit 1  │     │  FMA Unit 2  │     │ (overload?)  │
       └──────┬───────┘     └──────┬───────┘     └──────┬───────┘     └──────┬───────┘
              │                    │                    │                    │
     b[0][k..k+W]        b[1][k..k+W]         b[2][k..k+W]         b[3][k..k+W]
              ▼                    ▼                    ▼                    ▼
        sum[0] += ⋯          sum[1] += ⋯          sum[2] += ⋯          sum[3] += ⋯
```

---

#### Conclusion

To fully exploit the CPU:

* Use **blocking** to localize data into L1/L2 cache
* Use **SIMD width** for outer unrolling
* Use **FMA count** to choose how many accumulators to process in parallel
* **Avoid oversaturating** the pipeline by over-unrolling the inner loop

This design ensures high throughput, efficient cache use, and minimal stalls.


### Mathematical Explanation of Blocked Matrix Multiplication

Let $A \in \mathbb{K}^{m \times k}$ and $B \in \mathbb{K}^{k \times n}$ be two matrices over a field $\mathbb{K}$. The standard matrix product is defined by:

$$
C = A \cdot B \in \mathbb{K}^{m \times n}, \quad C_{ij} = \sum_{\ell=1}^k A_{i\ell} B_{\ell j}
$$

To optimize memory locality and parallelism, we partition $A$, $B$, and $C$ into **submatrices** (also called **blocks**). Suppose we fix a block size $b$ such that $m$, $n$, and $k$ are divisible by $b$. We define:

- $A_{pq} \in \mathbb{K}^{b \times b}$ as the block at row block $p$, column block $q$ of $A$
- $B_{qr} \in \mathbb{K}^{b \times b}$ as the block at row block $q$, column block $r$ of $B$
- $C_{pr} \in \mathbb{K}^{b \times b}$ as the block at row block $p$, column block $r$ of $C$

Then the matrix product becomes a **block matrix multiplication**:

$$
C_{pr} = \sum_{q=0}^{K/b - 1} A_{pq} \cdot B_{qr}
$$

This means: instead of computing each element $C_{ij}$ independently, we compute **entire submatrices** of $C$ at once by multiplying and accumulating submatrices of $A$ and $B$.

---

#### Example: Partitioning

Assume $A \in \mathbb{R}^{8 \times 8}$ and we choose a block size $b = 4$. Then $A$ is partitioned into:

$$
A =
\begin{pmatrix}
A_{00} & A_{01} \\
A_{10} & A_{11}
\end{pmatrix}, \quad \text{where each } A_{pq} \in \mathbb{R}^{4 \times 4}
$$

Similarly for $B$ and $C$.

The block multiplication is:

$$
C_{00} = A_{00} B_{00} + A_{01} B_{10} \\
C_{01} = A_{00} B_{01} + A_{01} B_{11} \\
C_{10} = A_{10} B_{00} + A_{11} B_{10} \\
C_{11} = A_{10} B_{01} + A_{11} B_{11}
$$

---

#### Algorithmic Advantage

By processing $C_{pr}$ block-by-block:

- Each access to memory is **reused efficiently** (spatial & temporal locality),
- SIMD microkernels can operate **on contiguous memory regions**,
- The CPU cache contains **relevant portions of $A$ and $B$** during the computation of each $C_{pr}$,
- The **outer loops** iterate over block indices, and the **inner loop** performs a SIMD-accelerated GEMM on small tiles.

---

#### SIMD Microkernel inside Each Block

Once $A_{pq}$ and $B_{qr}$ are loaded, we perform the inner multiplication:

$$
C_{pr}^{(q)} \mathrel{+}= A_{pq} \cdot B_{qr}
$$

using vectorized FMA instructions. These are **low-level dot products** between rows of $A_{pq}$ and columns of $B_{qr}$ (possibly transposed beforehand).

---

#### Final Summary

The full blocked matrix multiplication reads:

$$
C = A \cdot B = \begin{pmatrix}
\sum\limits_q A_{0q} B_{q0} & \cdots & \sum\limits_q A_{0q} B_{qR} \\
\vdots & \ddots & \vdots \\
\sum\limits_q A_{Pq} B_{q0} & \cdots & \sum\limits_q A_{Pq} B_{qR}
\end{pmatrix}
$$

This allows for a structured and parallel-friendly execution of matrix multiplication.

